In [ ]:
from pipeline import run_all
run_all()

In [1]:
# =============================================================================
# 1. IMPORTS & PATHS
# =============================================================================
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import timedelta
from scipy.stats import spearmanr, pearsonr
from launch_dashboard import generate_dashboard
from pipeline import read_features

import warnings
warnings.filterwarnings("ignore")

import lightgbm as lgb
from lightgbm import LGBMRegressor

# ── Indice de référence par univers (pour le beta/corrélation) ──
UNIVERSE_INDEX = {
    "spx":   "SPY",
    "cac40": "^FCHI",
    "dax":   "^GDAXI",
}

# =============================================================================
# ⚙️  CONFIGURATION PAR UNIVERS — modifie ici pour chaque univers
# =============================================================================
# Chaque univers peut avoir ses propres dates, hyperparamètres et fenêtres.
# Les clés disponibles :
#   start_date     : première date d'entraînement chargée
#   last_date      : dernière date chargée
#   holdout_start  : début du holdout (pas utilisé en walk-forward, seulement comme garde-fou)
#   train_years    : fenêtre d'entraînement glissante (en années)
#   test_months    : durée de chaque fold de test (en mois)
#   step_months    : pas entre deux folds (en mois)
#   embargo_bdays  : jours ouvrés d'embargo entre train et test
#   val_frac_wf    : fraction de validation dans le walk-forward
#   val_frac_final : fraction de validation pour le modèle final
#   history_start  : début du téléchargement Yahoo Finance
# =============================================================================

UNIVERSE_CONFIG = {
    "spx": {
        "start_date":     "2023-01-01",
        "last_date":      None,              # auto-détecté depuis la DB
        "holdout_start":  "2025-09-01",
        "train_years":    2,
        "test_months":    3,
        "step_months":    1,
        "embargo_bdays":  20,
        "val_frac_wf":    0.20,
        "val_frac_final": 0.10,
        "history_start":  "2019-01-01",
    },
    "cac40": {
        "start_date":     "2023-01-01",
        "last_date":      None,              # auto-détecté depuis la DB
        "holdout_start":  "2025-09-01",
        "train_years":    2,
        "test_months":    3,
        "step_months":    1,
        "embargo_bdays":  20,
        "val_frac_wf":    0.20,
        "val_frac_final": 0.10,
        "history_start":  "2019-01-01",
    },
    "dax": {
        "start_date":     "2023-01-01",
        "last_date":      None,              # auto-détecté depuis la DB
        "holdout_start":  "2025-09-01",
        "train_years":    2,
        "test_months":    3,
        "step_months":    1,
        "embargo_bdays":  20,
        "val_frac_wf":    0.20,
        "val_frac_final": 0.10,
        "history_start":  "2019-01-01",
    },
}

# ── Colonnes système : jamais des features (identiques pour tous les univers) ──
# Toute colonne à exclure PAR UNIVERS se définit dans DROP_FEATURES_XXX (cell 14/15/16)
SYSTEM_EXCLUDE = {
    "Date", "Ticker", "Universe",
    "Open", "High", "Low", "Close", "AdjClose", "Volume",
    "target_5d", "target_20d",
    "excess_target_5d", "excess_target_20d",
    "mkt_ret",
}

# ── Colonnes fondamentales (ne pas z-scorer en CS) ──
FUNDAMENTAL_COLS = {
    "eps_surprise", "eps_surprise_abs", "eps_beat",
    "eps_growth_qoq", "eps_surprise_ma4", "consecutive_beats",
    "revenue_growth_qoq", "revenue_growth_yoy",
    "net_margin", "debt_to_equity", "roa", "ocf_to_revenue",
}



In [2]:
# =============================================================================
# 2. LECTURE DES DONNÉES + TARGETS
# =============================================================================

def read_all_features(universes: list[str] = None) -> pd.DataFrame:
    """Lit les features depuis les DBs SQLite (remplace read_features_files)."""
    if universes is None:
        universes = ["spx", "cac40", "dax"]
    dfs = []
    for u in universes:
        df = read_features(u)
        df.columns = df.columns.str.strip()
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date", "Ticker"]).copy()
        df["Ticker"] = df["Ticker"].astype(str)
        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)
    out = out.sort_values(["Universe", "Ticker", "Date"], kind="mergesort").reset_index(drop=True)
    out = out.drop_duplicates(subset=["Universe", "Ticker", "Date"], keep="last").reset_index(drop=True)
    out["Universe"] = out["Universe"].astype("category")
    out["Ticker"]   = out["Ticker"].astype("category")
    return out


def add_forward_targets(
    df: pd.DataFrame, horizons=(5, 20), price_col="AdjClose"
) -> pd.DataFrame:
    df = df.sort_values(["Universe", "Ticker", "Date"], kind="mergesort").copy()
    if price_col not in df.columns:
        price_col = "Close" if "Close" in df.columns else None
        if price_col is None:
            raise ValueError("Missing AdjClose and Close")

    prices = pd.to_numeric(df[price_col], errors="coerce").where(lambda s: s > 0)
    df["_logp"] = np.log(prices)

    g = df.groupby(["Universe", "Ticker"], sort=False, observed=True)
    for h in horizons:
        df[f"target_{h}d"] = g["_logp"].shift(-h) - df["_logp"]

    return df.drop(columns=["_logp"])


def add_excess_targets(
    df: pd.DataFrame,
    horizons=(5, 20),
    zscore: bool = False,
    use_median: bool = True,
) -> tuple[pd.DataFrame, dict[int, pd.Series]]:
    df = df.copy()
    centers: dict[int, pd.Series] = {}
    for h in horizons:
        col = f"target_{h}d"
        if col not in df.columns:
            raise ValueError(f"Missing '{col}'")

        g = df.groupby(["Universe", "Date"], sort=False, observed=True)[col]
        center = g.transform("median" if use_median else "mean")
        ex = df[col] - center

        if zscore:
            sd = g.transform("std").replace(0, np.nan)
            ex = ex / sd

        df[f"excess_target_{h}d"] = ex
        centers[h] = center

    return df, centers

In [3]:
# =============================================================================
# 3. NORMALISATION CROSS-SECTIONNELLE
# =============================================================================

WINSOR_P_BY_UNI  = {"spx": 0.03, "cac40": 0.07, "dax": 0.07}
ZSCORE_MODE_BY_UNI = {"spx": "robust", "cac40": "robust", "dax": "robust"}
DO_WINSORIZE  = True
DO_CS_SCALE   = True
NO_WINSOR_COLS = {"z_price_5", "z_price_10",
                  "macro_vix", "macro_vix_chg_20", "macro_term_spread"}
NO_ZSCORE_COLS = {"z_price_5", "z_price_10",
                  "macro_vix", "macro_vix_chg_20", "macro_term_spread"}
CS_GROUP_DATE  = ["Date"]


def winsorize_cs_one_universe(df_u: pd.DataFrame, cols: list[str], p: float) -> pd.DataFrame:
    """Winsorisation cross-sectionnelle par date."""
    out = df_u.copy()

    def _winsor_day(g):
        n = len(g)
        if n < 10:
            return g
        k = max(1, int(np.floor(p * n)))
        p_eff = k / n
        lo = g[cols].quantile(p_eff)
        hi = g[cols].quantile(1 - p_eff)
        g[cols] = g[cols].clip(lo, hi, axis=1)
        return g

    return out.groupby("Date", group_keys=False, sort=False).apply(_winsor_day)


def zscore_cs_one_universe(df_u: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df_u.copy()
    g  = out.groupby(CS_GROUP_DATE, sort=False, observed=True)
    mu = g[cols].transform("mean")
    sd = g[cols].transform("std").replace(0, np.nan)
    out[cols] = (out[cols] - mu) / sd
    return out


def robust_zscore_cs_one_universe(df_u: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """MAD-based robust z-score (optimisé : un seul groupby)."""
    out = df_u.copy()
    g   = out.groupby("Date", sort=False, observed=True)
    med = g[cols].transform("median")
    # MAD = median des |xi - median|
    abs_dev = (out[cols] - med).abs()
    mad     = abs_dev.groupby(out["Date"], sort=False).transform("median")
    scale   = (1.4826 * mad).replace(0, np.nan)
    # Fallback sur std si MAD nul
    std     = g[cols].transform("std").replace(0, np.nan)
    scale   = scale.where(scale.notna(), std)
    out[cols] = (out[cols] - med) / scale
    return out


def apply_cs_transforms_by_universe(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    parts = []
    for u, df_u in df.groupby("Universe", observed=True, sort=False):
        df_u = df_u.copy()
        cols_winsor = [c for c in cols if c not in NO_WINSOR_COLS]
        cols_zscore = [c for c in cols if c not in NO_ZSCORE_COLS]

        if DO_WINSORIZE and cols_winsor:
            p = WINSOR_P_BY_UNI.get(str(u), 0.01)
            df_u = winsorize_cs_one_universe(df_u, cols_winsor, p=p)

        if DO_CS_SCALE and cols_zscore:
            mode = ZSCORE_MODE_BY_UNI.get(str(u), "zscore")
            if mode == "robust":
                df_u = robust_zscore_cs_one_universe(df_u, cols_zscore)
            else:
                df_u = zscore_cs_one_universe(df_u, cols_zscore)

        parts.append(df_u)

    return (
        pd.concat(parts, ignore_index=True)
        .sort_values(["Universe", "Ticker", "Date"], kind="mergesort")
        .reset_index(drop=True)
    )

In [4]:
# =============================================================================
# 4b. PRÉPARATION DES FEATURES PAR UNIVERS
# =============================================================================
# Le parquet contient déjà : tech + fund + temporal + interactions + yf_*
# Cette fonction applique uniquement la normalisation CS sur les features
# techniques de base — exactement comme avant le refactor pipeline.
#
# CS norm appliquée sur  : tech de base (mom_*, rv_*, dist_*, slope_*, etc.)
# CS norm NON appliquée  : yf_*, delta_*, ma_*, interactions, fondamentaux
#
# Utilisation :
#   df_ready, all_feature_cols = prepare_feature_df(df)
#   feature_cols = [c for c in all_feature_cols if c not in EXCLUDE_XXX]
# =============================================================================

# Préfixes et noms exclus de la CS norm (même comportement qu'avant)
NO_CS_NORM_PREFIXES = ("yf_", "delta_", "ma_", "macro_")
NO_CS_NORM_EXACT    = {"mom120_div_rv60", "mom20_div_mom120"}


def prepare_feature_df(df: pd.DataFrame) -> tuple:
    """
    Applique la normalisation cross-sectionnelle sur les features techniques
    de base uniquement, et retourne (df_ready, all_feature_cols).

    Exclus de la CS norm (traités comme les fondamentaux) :
      - yf_*          : déjà en échelle naturelle (RSI 0-100, ratios, etc.)
      - delta_*, ma_* : variations et moyennes mobiles
      - interactions  : mom120_div_rv60, mom20_div_mom120
      - fondamentaux  : eps_*, revenue_*, net_margin, etc.
    """
    # 1. Toutes les colonnes numériques hors système
    base_cols = [
        c for c in df.columns
        if c not in SYSTEM_EXCLUDE and pd.api.types.is_numeric_dtype(df[c])
    ]

    # 2. Colonnes à CS-normaliser = tech de base uniquement
    def _needs_cs_norm(col):
        if col in FUNDAMENTAL_COLS:
            return False
        if col in NO_CS_NORM_EXACT:
            return False
        if any(col.startswith(p) for p in NO_CS_NORM_PREFIXES):
            return False
        return True

    tech_cs_cols = [c for c in base_cols if _needs_cs_norm(c)]
    fund_cols    = [c for c in base_cols if c in FUNDAMENTAL_COLS]

    # 3. CS normalisation sur tech de base uniquement
    df_out = apply_cs_transforms_by_universe(df, tech_cs_cols)

    # 4. Statistiques
    n_yf    = len([c for c in base_cols if c.startswith("yf_")])
    n_temp  = len([c for c in base_cols if c.startswith("delta_") or c.startswith("ma_")])
    n_inter = len([c for c in base_cols if c in NO_CS_NORM_EXACT])
    n_macro = len([c for c in base_cols if c.startswith("macro_")])
    print(f"  Base: {len(base_cols)} (tech_cs={len(tech_cs_cols)}, fund={len(fund_cols)}, macro={n_macro})")
    print(f"  Temporal: {n_temp}  Interactions: {n_inter}  Yahoo: {n_yf}  Macro: {n_macro}")
    print(f"  => {len(base_cols)} features disponibles au total")

    return df_out, base_cols


In [5]:
# =============================================================================
# 5. MÉTRIQUES DE RANKING  (vectorisé via groupby)
# =============================================================================

def per_date_spearman_ic(
    df_pred: pd.DataFrame, pred_col="pred", y_col="y_true", min_obs: int = 5
) -> float:
    """IC Spearman moyen par date — vectorisé."""
    def _ic(g):
        sub = g[[pred_col, y_col]].dropna()
        if sub[pred_col].nunique() < min_obs or sub[y_col].nunique() < min_obs:
            return np.nan
        return sub[pred_col].corr(sub[y_col], method="spearman")

    ics = df_pred.groupby("Date", sort=False).apply(_ic).dropna()
    return float(ics.mean()) if len(ics) else np.nan


def per_date_pearson_ic(
    df_pred: pd.DataFrame, pred_col="pred", y_col="y_true", min_obs: int = 5
) -> float:
    """IC Pearson moyen par date — vectorisé."""
    def _ic(g):
        sub = g[[pred_col, y_col]].dropna()
        if sub[pred_col].nunique() < min_obs or sub[y_col].nunique() < min_obs:
            return np.nan
        return sub[pred_col].corr(sub[y_col], method="pearson")

    ics = df_pred.groupby("Date", sort=False).apply(_ic).dropna()
    return float(ics.mean()) if len(ics) else np.nan


def per_date_long_short_spread(
    df_pred: pd.DataFrame, q=0.25, pred_col="pred", y_col="y_true"
) -> float:
    spreads = []
    for _, g in df_pred.groupby("Date", sort=False):
        g = g.dropna(subset=[pred_col, y_col])
        n = len(g)
        if n < 10:
            continue
        g = g.sort_values(pred_col, ascending=False)
        k = max(1, int(np.floor(q * n)))
        spreads.append(float(g.head(k)[y_col].mean() - g.tail(k)[y_col].mean()))
    return float(np.mean(spreads)) if spreads else np.nan


def compute_feature_ic(
    df_u: pd.DataFrame, feature_cols: list[str], target_col: str, min_obs: int = 20
) -> pd.DataFrame:
    """
    IC Spearman par feature, calculé cross-sectionnellement par date.
    Vectorisé : un seul groupby par feature.
    """
    rows = []
    for col in feature_cols:
        def _ic(g, col=col):
            sub = g[[col, target_col]].dropna()
            if len(sub) < min_obs:
                return np.nan
            return sub[col].corr(sub[target_col], method="spearman")

        ics = df_u.groupby("Date", sort=False).apply(_ic).dropna()
        if len(ics) == 0:
            continue
        ic_arr = ics.values
        rows.append({
            "feature":         col,
            "ic_mean":         ic_arr.mean(),
            "ic_std":          ic_arr.std(),
            "ic_ir":           ic_arr.mean() / ic_arr.std() if ic_arr.std() > 0 else 0,
            "ic_positive_pct": (ic_arr > 0).mean(),
        })
    return pd.DataFrame(rows).sort_values("ic_ir", ascending=False).reset_index(drop=True)

In [6]:
# =============================================================================
# 5b. HELPERS INTERNES — Feature Selection
# =============================================================================
# Fonctions privées partagées par cell_06c et cell_06d. Ne pas appeler directement.
# =============================================================================

def _score_feature_set(
    fold_arrays: list,
    feature_cols: list,
    lgb_params: dict,
    early_stopping_rounds: int,
    seed: int,
    metric: str,
) -> float:
    """
    Entraîne sur chaque fold, POOL tous les IC journaliers de tous les folds.
    IC-IR calculé sur ~400 daily ICs au lieu de 8 fold means → score stable.
    """
    params = dict(lgb_params)
    eval_metric = "l1" if params.get("objective") == "regression_l1" else "rmse"
    all_cols = fold_arrays[0]["all_cols"]
    col_idx  = [i for i, c in enumerate(all_cols) if c in set(feature_cols)]

    all_daily_ics = []   # poolé : tous les IC journaliers de tous les folds
    for fa in fold_arrays:
        Xtr  = fa["Xtr"][:, col_idx]
        Xval = fa["Xval"][:, col_idx]
        Xte  = fa["Xte"][:, col_idx]

        model = LGBMRegressor(random_state=seed, verbose=-1, n_jobs=-1, **params)
        if early_stopping_rounds > 0:
            model.fit(
                Xtr, fa["ytr"],
                eval_set=[(Xval, fa["yval"])],
                eval_metric=eval_metric,
                callbacks=[lgb.early_stopping(
                    stopping_rounds=early_stopping_rounds, verbose=False
                )],
            )
        else:
            model.fit(Xtr, fa["ytr"])
        pred = model.predict(Xte).astype(float)
        df_pred = pd.DataFrame({
            "Date":   fa["te_dates"],
            "y_true": fa["yte"],
            "pred":   pred,
        })
        daily_ics = (
            df_pred.groupby("Date")
            .apply(lambda g: (
                g[["pred", "y_true"]].dropna()
                .pipe(lambda s: float(s["pred"].corr(s["y_true"], method="spearman"))
                      if len(s) >= 5 else np.nan)
            ))
            .dropna()
            .values
        )
        all_daily_ics.extend(daily_ics.tolist())

    if len(all_daily_ics) < 10:
        return -np.inf
    arr = np.array(all_daily_ics)
    if metric == "ic_mean":
        return float(arr.mean())
    else:
        return float(arr.mean() / arr.std()) if arr.std() > 0 else 0.0


def _prepare_fold_arrays_fs(
    df_u: pd.DataFrame,
    folds: pd.DataFrame,
    all_feature_cols: list,
    target_col: str,
    val_frac: float,
    min_train: int,
    min_val: int,
    min_test: int,
) -> list:
    """Pré-calcule les matrices numpy pour tous les folds (toutes features d'un coup)."""
    fold_arrays = []
    for _, fold in folds.iterrows():
        tr0 = pd.Timestamp(fold["train_start"])
        tr1 = pd.Timestamp(fold["train_end"])
        te0 = pd.Timestamp(fold["test_start"])
        te1 = pd.Timestamp(fold["test_end"])

        df_train_all = df_u.loc[
            (df_u["Date"] >= tr0) & (df_u["Date"] <= tr1)
        ].dropna(subset=[target_col])
        df_test = df_u.loc[
            (df_u["Date"] >= te0) & (df_u["Date"] <= te1)
        ].dropna(subset=[target_col])

        if len(df_train_all) < min_train or len(df_test) < min_test:
            continue
        unique_dates = np.sort(df_train_all["Date"].unique())
        if len(unique_dates) < 20:
            continue

        k = max(1, int(np.floor(val_frac * len(unique_dates))))
        val_start = unique_dates[-k]
        is_val    = df_train_all["Date"] >= val_start
        df_tr     = df_train_all[~is_val]
        df_val    = df_train_all[is_val]

        if len(df_tr) < min_train or len(df_val) < min_val:
            continue

        fold_arrays.append(dict(
            all_cols   = all_feature_cols,
            Xtr        = df_tr[all_feature_cols].to_numpy(dtype=np.float32),
            Xval       = df_val[all_feature_cols].to_numpy(dtype=np.float32),
            Xte        = df_test[all_feature_cols].to_numpy(dtype=np.float32),
            ytr        = df_tr[target_col].to_numpy(dtype=np.float32),
            yval       = df_val[target_col].to_numpy(dtype=np.float32),
            yte        = df_test[target_col].to_numpy(dtype=np.float32),
            te_dates   = df_test["Date"].values,
            te_tickers = df_test["Ticker"].astype(str).values,
        ))
    return fold_arrays


def _compute_corr_matrix(
    df_u: pd.DataFrame,
    feature_cols: list,
) -> pd.DataFrame:
    """Matrice de corrélation absolue moyenne sur les dates (cross-sectional)."""
    # Corrélation sur toutes les données disponibles — simple et rapide
    df_feat = df_u[feature_cols].copy()
    return df_feat.corr(method="spearman").abs()


def _get_correlated_with(
    col: str,
    selected: list,
    corr_matrix: pd.DataFrame,
    corr_threshold: float,
) -> str | None:
    """Retourne le nom de la feature déjà sélectionnée la plus corrélée, ou None."""
    for sel in selected:
        if sel in corr_matrix.index and col in corr_matrix.columns:
            if corr_matrix.loc[sel, col] >= corr_threshold:
                return sel
    return None


def _compute_ic_stats(
    df_u: pd.DataFrame,
    all_feature_cols: list,
    target_col: str,
) -> list:
    """
    Calcule IC-IR et couverture pour chaque feature.
    Retourne une liste de (col, ic_ir, cov) triee par IC-IR decroissant.
    Partage entre find_best_seed_top_n et forward_feature_selection.
    """
    df_train = df_u.dropna(subset=[target_col])
    rows = []
    for col in all_feature_cols:
        def _ic(g, col=col):
            sub = g[[col, target_col]].dropna()
            return sub[col].corr(sub[target_col], method="spearman") if len(sub) >= 20 else np.nan
        ics = df_train.groupby("Date", sort=False).apply(_ic).dropna().values
        cov = float(df_u[col].notna().mean())
        if len(ics) >= 2:
            icir = ics.mean() / ics.std() if ics.std() > 0 else 0.0
            rows.append((col, float(icir), cov))
    rows.sort(key=lambda x: x[1], reverse=True)
    return rows

In [7]:
# =============================================================================
# 5c. OUTILS — Analyse et calibration du seed
# =============================================================================
#
# A) show_feature_table : tableau IC-IR / IC mean / couverture
#
# B) find_best_seed_top_n : teste plusieurs valeurs de seed_top_n
#    avec les memes filtres que forward_feature_selection
#    (couverture >= seed_min_cov, correlation >= corr_threshold)
#    Retourne : (best_n, seed_features)
#      - best_n        : int  — seed_top_n optimal
#      - seed_features : list — features du seed, à passer à
#                        forward_feature_selection(seed_features=...)
#
#    Usage (deux niveaux de corrélation) :
#        best_n, seed_cols = find_best_seed_top_n(..., corr_threshold=0.70)
#        selected, history = forward_feature_selection(
#            ..., seed_features=seed_cols, corr_threshold=0.90)
#
#    Workflow :
#        best_n = find_best_seed_top_n(df_cs_cac, folds_cac, 'cac40', ...)
#        selected_cac, history_cac = forward_feature_selection(
#            ..., seed_top_n=best_n, ...)
# =============================================================================

def show_feature_table(
    df: pd.DataFrame,
    feature_cols: list,
    target_col: str,
    min_obs: int = 20,
) -> pd.DataFrame:
    rows = []
    for col in feature_cols:
        def _ic(g, col=col):
            sub = g[[col, target_col]].dropna()
            if len(sub) < min_obs:
                return np.nan
            return sub[col].corr(sub[target_col], method="spearman")

        ics = df.groupby("Date", sort=False).apply(_ic).dropna().values
        if len(ics) == 0:
            # IC = NaN partout (broadcast macro, ou couverture trop faible)
            rows.append({
                "feature":    col,
                "ic_mean":    0.0,
                "ic_std":     0.0,
                "ic_ir":      0.0,
                "ic_pos_pct": 50.0,
                "cov_pct":    round(df[col].notna().mean() * 100, 1),
            })
            continue
        rows.append({
            "feature":    col,
            "ic_mean":    round(ics.mean(), 4),
            "ic_std":     round(ics.std(),  4),
            "ic_ir":      round(ics.mean() / ics.std(), 4) if ics.std() > 0 else 0.0,
            "ic_pos_pct": round((ics > 0).mean() * 100, 1),
            "cov_pct":    round(df[col].notna().mean() * 100, 1),
        })

    tbl = pd.DataFrame(rows).sort_values("ic_ir", ascending=False).reset_index(drop=True)
    W = 90
    print("=" * W)
    print(f"  FEATURE TABLE  ({len(tbl)} features)")
    print("=" * W)
    print(f"  {'Feature':<32} {'IC-IR':>7} {'IC mean':>8} {'IC std':>7} {'IC%+':>6} {'Cov%':>6}")
    print("-" * W)
    for _, r in tbl.iterrows():
        print(f"  {r['feature']:<32} {r['ic_ir']:>+7.3f} {r['ic_mean']:>+8.4f} "
              f"{r['ic_std']:>7.4f} {r['ic_pos_pct']:>5.1f}% {r['cov_pct']:>5.1f}%")
    print("=" * W)
    return tbl


# ── B. Forward feature selection ─────────────────────────────────────────────


def find_best_seed_top_n(
    df_cs: pd.DataFrame,
    folds: pd.DataFrame,
    universe: str,
    all_feature_cols: list,
    target_col: str,
    lgb_params: dict,
    metric: str = "ic_mean",
    seed_min_cov: float = 0.80,
    corr_threshold: float = 0.80,
    seed_exclude: 'set | None' = None,            # features exclues du seed (ex: FUNDAMENTAL_COLS)
    corr_matrix: 'pd.DataFrame | None' = None,   # pré-calculée pour éviter double calcul
    ic_stats: 'list | None' = None,               # pré-calculée pour éviter double calcul
    fold_arrays: 'list | None' = None,            # pré-calculée pour éviter double calcul
    ns_to_test: list = None,
    early_stopping_rounds: int = 100,
    val_frac: float = 0.15,
    min_train: int = 300,
    min_val: int = 50,
    min_test: int = 50,
    seed: int = 42,
) -> tuple[int, list]:
    """
    Trouve le seed_top_n optimal à injecter dans forward_feature_selection.

    Applique exactement les mêmes filtres que forward_feature_selection :
      - couverture >= seed_min_cov
      - pré-filtrage corrélation >= corr_threshold

    Teste chaque valeur de ns_to_test en mesurant le vrai IC walk-forward,
    et retourne le seed_top_n qui maximise la métrique.

    Usage :
        best_n = find_best_seed_top_n(df_cs_cac, folds_cac, "cac40", ...)
        selected_cac, history_cac = forward_feature_selection(
            ..., seed_top_n=best_n, seed_min_cov=0.80, corr_threshold=0.80, ...
        )
    """
    metric_lbl = "IC mean" if metric == "ic_mean" else "IC-IR"
    if ns_to_test is None:
        ns_to_test = [3, 5, 8, 10, 15, 20]

    print(f"\n{'-'*76}")
    print(f"  SEED CALIBRATION — {universe.upper()}")
    print(f"  metric={metric} | cov_min={seed_min_cov:.0%} | corr_max={corr_threshold:.0%} | n_test={ns_to_test}")
    print(f"{'-'*76}")

    df_u = df_cs[df_cs["Universe"].astype(str) == universe].copy()
    df_u = df_u.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    df_train = df_u.dropna(subset=[target_col])

    # ── IC-IR + couverture (identique au seed de forward_feature_selection) ──
    if ic_stats is None:
        ic_rows = []
        for col in all_feature_cols:
            def _ic(g, col=col):
                sub = g[[col, target_col]].dropna()
                return sub[col].corr(sub[target_col], method="spearman") if len(sub) >= 20 else np.nan
            ics = df_train.groupby("Date", sort=False).apply(_ic).dropna().values
            cov = float(df_u[col].notna().mean())
            if len(ics) >= 2:
                icir = ics.mean() / ics.std() if ics.std() > 0 else 0.0
                ic_rows.append((col, float(icir), cov))
        ic_rows.sort(key=lambda x: x[1], reverse=True)
    else:
        ic_rows = ic_stats

    # ── Pré-filtrage corrélation (identique à forward_feature_selection) ──
    if corr_matrix is None:
        corr_matrix = _compute_corr_matrix(df_u, all_feature_cols)
    kept_set = set()
    n_corr_excl = 0
    for col, icir, cov in ic_rows:
        if any(corr_matrix.loc[k, col] >= corr_threshold
               for k in kept_set if k in corr_matrix.index and col in corr_matrix.columns):
            n_corr_excl += 1
        else:
            kept_set.add(col)

    # ── Features éligibles au seed ──
    _seed_excl = set(seed_exclude) if seed_exclude else set()
    seed_eligible = [
        col for col, icir, cov in ic_rows
        if col in kept_set
        and cov >= seed_min_cov
        and col not in _seed_excl
    ]
    if _seed_excl:
        n_excl_fund = sum(1 for col, _, _ in ic_rows if col in _seed_excl and col in kept_set)
        print(f"  seed_exclude : {n_excl_fund} fondamentales exclues du seed (disponibles en forward)")
    n_elig = len(seed_eligible)
    n_excl_fund_total = len([col for col, _, _ in ic_rows if col in _seed_excl]) if _seed_excl else 0
    print(f"  {n_elig}/{len(all_feature_cols)} features eligibles  ({n_corr_excl} exclues corr>={corr_threshold:.0%}, {len(all_feature_cols)-n_elig-n_corr_excl-n_excl_fund_total} sous cov<{seed_min_cov:.0%}, {n_excl_fund_total} seed_exclude)")

    ns_to_test = [n for n in ns_to_test if n <= n_elig]
    if not ns_to_test:
        fallback = min(5, n_elig)
        print(f"  Aucune valeur testable — retourne {fallback}")
        return fallback, seed_eligible[:fallback]

    # ── Pré-calculer les fold arrays ──
    if fold_arrays is None:
        fold_arrays = _prepare_fold_arrays_fs(
            df_u, folds, all_feature_cols, target_col,
            val_frac, min_train, min_val, min_test,
        )
    if not fold_arrays:
        print("  [ERROR] Aucun fold valide.")
        return ns_to_test[0], seed_eligible[:ns_to_test[0]]


    print(f"  {'N':>4}  {'Seed (top-3)':<42} {metric_lbl:>8}  {'Δ':>7}")
    print(f"  {'-'*76}")

    results = []
    prev = None
    for n in ns_to_test:
        cols = seed_eligible[:n]
        score = _score_feature_set(
            fold_arrays, cols, lgb_params, early_stopping_rounds, seed, metric
        )
        delta = f"{score - prev:>+7.4f}" if prev is not None and not np.isinf(score) else "      -"
        top3  = ", ".join(cols[:3]) + ("..." if n > 3 else "")
        sc_s  = f"{score:>+8.4f}" if not np.isinf(score) else "    -inf"
        print(f"  {n:>4}  {top3:<42} {sc_s}  {delta}")
        if not np.isinf(score):
            results.append((n, score))
            prev = score

    if not results:
        return ns_to_test[0], seed_eligible[:ns_to_test[0]]

    best_n, best_score = max(results, key=lambda x: x[1])
    best_seed_cols = seed_eligible[:best_n]

    print(f"\n  → seed_top_n = {best_n}  ({metric_lbl} = {best_score:+.4f})")
    print(f"{'-'*76}\n")
    return best_n, best_seed_cols


In [8]:
# =============================================================================
# 5d. FEATURE SELECTION — Diagnose & Hill-Climb
# =============================================================================
#
# Approche pragmatique : partir d'un set trouvé manuellement ou par
# permutation, et tester systématiquement chaque ajout/retrait.
#
# Le scoring utilise ~400 daily ICs poolés (pas 8 fold means) → stable.
# =============================================================================


def diagnose_feature_set(
    df_cs: pd.DataFrame,
    folds: pd.DataFrame,
    universe: str,
    current_features: list,
    all_features: list,
    target_col: str,
    lgb_params: dict,
    metric: str = "ic_ir",
    fold_arrays: 'list | None' = None,
    early_stopping_rounds: int = 100,
    val_frac: float = 0.15,
    min_train: int = 300,
    min_val: int = 50,
    min_test: int = 50,
    seed: int = 42,
    max_rounds: int = 20,
) -> tuple:
    """
    Hill-climbing depuis un set existant.

    1) Calcule le score baseline du set actuel
    2) Scan complet : teste retrait et ajout de chaque feature
    3) Applique itérativement la meilleure modification
    4) Répète jusqu'à convergence

    Returns: (improved_features, history_df)
    """
    assert metric in ("ic_ir", "ic_mean")
    metric_lbl = "IC mean" if metric == "ic_mean" else "IC-IR"

    print(f"\n{'='*80}")
    print(f"  DIAGNOSE & HILL-CLIMB — {universe.upper()}")
    print(f"  Current: {len(current_features)} features  |  Pool: {len(all_features)}  |  metric={metric}")
    print(f"{'='*80}")

    df_u = df_cs[df_cs["Universe"].astype(str) == universe].copy()
    df_u = df_u.sort_values(["Ticker", "Date"]).reset_index(drop=True)

    if fold_arrays is None:
        fold_arrays = _prepare_fold_arrays_fs(
            df_u, folds, all_features, target_col,
            val_frac, min_train, min_val, min_test,
        )
    if not fold_arrays:
        print("  [ERROR] Aucun fold valide.")
        return current_features, pd.DataFrame()

    selected = list(current_features)

    # ── Baseline ──
    current_score = _score_feature_set(
        fold_arrays, selected, lgb_params,
        early_stopping_rounds, seed, metric
    )
    print(f"\n  Baseline : {len(selected)} features  |  {metric_lbl} = {current_score:+.4f}")

    # ── Scan complet : retraits ──
    print(f"\n  ── Scan retraits ({len(selected)} features) ──\n")
    print(f"  {'Feature':<35} {metric_lbl+' sans':>12}  {'Δ':>8}")
    print(f"  {'-'*62}")
    remove_results = []
    for col in sorted(selected):
        subset = [c for c in selected if c != col]
        sc = _score_feature_set(fold_arrays, subset, lgb_params, early_stopping_rounds, seed, metric)
        delta = sc - current_score
        tag = "  ← RETIRER" if delta > 0.005 else ""
        sc_s = f"{sc:>+12.4f}" if not np.isinf(sc) else "        -inf"
        print(f"  {col:<35} {sc_s}  {delta:>+8.4f}{tag}")
        remove_results.append((col, sc, delta))
    remove_results.sort(key=lambda x: x[2], reverse=True)

    # ── Scan complet : ajouts ──
    not_in_set = [c for c in all_features if c not in set(selected)]
    print(f"\n  ── Scan ajouts ({len(not_in_set)} candidates) ──\n")
    print(f"  {'Feature':<35} {metric_lbl+' avec':>12}  {'Δ':>8}")
    print(f"  {'-'*62}")
    add_results = []
    for col in not_in_set:
        sc = _score_feature_set(fold_arrays, selected + [col], lgb_params, early_stopping_rounds, seed, metric)
        delta = sc - current_score
        tag = "  ← AJOUTER" if delta > 0.005 else ""
        sc_s = f"{sc:>+12.4f}" if not np.isinf(sc) else "        -inf"
        print(f"  {col:<35} {sc_s}  {delta:>+8.4f}{tag}")
        add_results.append((col, sc, delta))
    add_results.sort(key=lambda x: x[2], reverse=True)

    # ── Hill-climbing ──
    print(f"\n  ── Hill-climbing ──")
    print(f"\n  {'Rnd':>4}  {'Action':<42} {metric_lbl:>8}  {'Δ':>8}  {'N':>4}")
    print(f"  {'-'*76}")

    history = []
    for rnd in range(max_rounds):
        best_action = None
        best_delta  = 0.0

        for col in list(selected):
            subset = [c for c in selected if c != col]
            sc = _score_feature_set(fold_arrays, subset, lgb_params, early_stopping_rounds, seed, metric)
            delta = sc - current_score
            if delta > best_delta:
                best_action = ("remove", col, sc, delta)
                best_delta = delta

        not_in = [c for c in all_features if c not in set(selected)]
        for col in not_in:
            sc = _score_feature_set(fold_arrays, selected + [col], lgb_params, early_stopping_rounds, seed, metric)
            delta = sc - current_score
            if delta > best_delta:
                best_action = ("add", col, sc, delta)
                best_delta = delta

        if best_action is None:
            print(f"  Rnd {rnd+1:>2}  Stop : aucune modification n'améliore le score")
            break

        action, col, new_score, delta = best_action
        if action == "remove":
            selected.remove(col)
            label = f"−{col}"
        else:
            selected.append(col)
            label = f"+{col}"

        current_score = new_score
        print(f"  Rnd {rnd+1:>2}  {label:<42} {current_score:>+8.4f}  "
              f"{delta:>+8.4f}  [{len(selected)}]")
        history.append({
            "round": rnd + 1, "action": action, "feature": col,
            metric_lbl: round(current_score, 4),
            "delta": round(delta, 4), "n_features": len(selected),
        })

    history_df = pd.DataFrame(history)

    print(f"\n  {'─'*76}")
    print(f"  RÉSULTAT : {len(selected)} features  |  {metric_lbl} = {current_score:+.4f}")
    print(f"  Selected : {selected}")

    not_selected = [c for c in all_features if c not in set(selected)]
    print(f"\n  EXCLUDE ({len(not_selected)}) :")
    print("    " + ",\n    ".join(repr(c) for c in not_selected))
    print(f"  {'─'*76}\n")

    return selected, history_df


In [9]:
# =============================================================================
# 6. LGBM BUILDER
# =============================================================================

def build_lgbm_regressor(
    seed: int = 42, verbose: int = -1, n_jobs: int = -1, **params
) -> LGBMRegressor:
    base = dict(objective="regression", random_state=seed, n_jobs=n_jobs, verbose=verbose)
    base.update(params)
    return LGBMRegressor(**base)

In [10]:
# =============================================================================
# 7. WALK-FORWARD FOLDS
# =============================================================================

def make_walkforward_folds(
    df: pd.DataFrame,
    start_date="2020-01-01",
    holdout_start="2025-01-01",
    train_years=5,
    test_months=6,
    step_months=1,
    embargo_bdays=5,
    date_col="Date",
) -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)

    dmin    = max(pd.Timestamp(start_date), df[date_col].min())
    holdout = pd.Timestamp(holdout_start)
    dates   = pd.Index(df[date_col].unique()).sort_values()

    def next_available_date(dt):
        pos = dates.searchsorted(pd.Timestamp(dt), side="left")
        return dates[pos] if pos < len(dates) else None

    def prev_available_date(dt):
        pos = dates.searchsorted(pd.Timestamp(dt), side="right") - 1
        return dates[pos] if pos >= 0 else None

    anchor_dt     = pd.Timestamp(dmin) + pd.DateOffset(years=train_years)
    first_anchor  = pd.Timestamp(anchor_dt.year, anchor_dt.month, 1)

    folds = []
    t = first_anchor
    while t < holdout:
        test_start = next_available_date(t)
        if test_start is None or test_start >= holdout:
            break

        raw_test_end = t + pd.DateOffset(months=test_months)
        test_end     = prev_available_date(min(raw_test_end, holdout))
        if test_end is None or test_end < test_start:
            break

        pos_ts = dates.searchsorted(test_start, side="left")
        pos_te = pos_ts - embargo_bdays - 1
        if pos_te <= 0:
            break
        train_end = dates[pos_te]

        train_start_raw = pd.Timestamp(test_start) - pd.DateOffset(years=train_years)
        train_start     = next_available_date(train_start_raw)
        if train_start is None or train_end < train_start:
            break

        folds.append({
            "train_start":   pd.Timestamp(train_start).normalize(),
            "train_end":     pd.Timestamp(train_end).normalize(),
            "test_start":    pd.Timestamp(test_start).normalize(),
            "test_end":      pd.Timestamp(test_end).normalize(),
            "embargo_bdays": embargo_bdays,
        })
        t = t + pd.DateOffset(months=step_months)

    return pd.DataFrame(folds)

In [11]:
# =============================================================================
# 8. WALK-FORWARD RUNNER — POINT-IN-TIME (affichage condensé 1 ligne/fold)
# =============================================================================

def _make_skip_row(fold_id, tr0, tr1, te0, te1, status, **extra):
    row = {
        "fold_id": fold_id, "train_start": tr0.date(), "train_end": tr1.date(),
        "test_start": te0.date(), "test_end": te1.date(), "status": status,
        "n_train": 0, "n_val": 0, "n_test": 0,
        "best_iteration": np.nan,
        "val_ic_spearman": np.nan, "val_ic_pearson": np.nan, "val_ls_spread": np.nan,
        "ic_spearman":     np.nan, "ic_pearson":     np.nan, "ls_spread":     np.nan,
        "val_metric_last": np.nan,
    }
    row.update(extra)
    return row


def walkforward_lgbm_rank(
    df_cs: pd.DataFrame,
    folds: pd.DataFrame,
    universe: str,
    feature_cols: list,
    target_col: str,
    q_spread: float = 0.25,
    seed: int = 42,
    val_frac: float = 0.15,
    early_stopping_rounds: int = 100,
    lgb_params: dict | None = None,
    verbose_lgb: int = -1,
    min_train: int = 300,
    min_val: int = 50,
    min_test: int = 50,
):
    all_importances = []
    df_u = df_cs[df_cs["Universe"].astype(str) == universe].copy()
    df_u = df_u.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    fold_rows, all_test_preds = [], []

    for i, f in folds.iterrows():
        tr0, tr1 = pd.Timestamp(f["train_start"]), pd.Timestamp(f["train_end"])
        te0, te1 = pd.Timestamp(f["test_start"]),  pd.Timestamp(f["test_end"])

        df_train_all = df_u.loc[
            (df_u["Date"] >= tr0) & (df_u["Date"] <= tr1)
        ].dropna(subset=[target_col]).copy()

        if len(df_train_all) < min_train:
            fold_rows.append(_make_skip_row(int(i), tr0, tr1, te0, te1,
                                            "skipped_not_enough_train", n_train=len(df_train_all)))
            print(f"  Fold {int(i):>2} | SKIPPED (not enough train: {len(df_train_all)})")
            continue

        df_test = df_u.loc[
            (df_u["Date"] >= te0) & (df_u["Date"] <= te1)
        ].dropna(subset=[target_col]).copy()

        if len(df_test) < min_test:
            fold_rows.append(_make_skip_row(int(i), tr0, tr1, te0, te1,
                                            "skipped_not_enough_test",
                                            n_train=len(df_train_all), n_test=len(df_test)))
            print(f"  Fold {int(i):>2} | SKIPPED (not enough test: {len(df_test)})")
            continue

        unique_dates_tr = np.sort(df_train_all["Date"].unique())
        if len(unique_dates_tr) < 20:
            fold_rows.append(_make_skip_row(int(i), tr0, tr1, te0, te1,
                                            "skipped_not_enough_train_dates", n_train=len(df_train_all)))
            print(f"  Fold {int(i):>2} | SKIPPED (not enough train dates)")
            continue

        k = max(1, int(np.floor(val_frac * len(unique_dates_tr))))
        val_start_date = unique_dates_tr[-k]
        is_val = df_train_all["Date"] >= val_start_date
        df_tr  = df_train_all[~is_val]
        df_val = df_train_all[ is_val]

        if len(df_tr) < min_train or len(df_val) < min_val:
            fold_rows.append(_make_skip_row(int(i), tr0, tr1, te0, te1,
                                            "skipped_train_val_too_small",
                                            n_train=len(df_tr), n_val=len(df_val)))
            print(f"  Fold {int(i):>2} | SKIPPED (train/val too small)")
            continue

        Xtr  = df_tr[feature_cols].to_numpy(dtype=np.float32)
        ytr  = df_tr[target_col].to_numpy(dtype=np.float32)
        Xval = df_val[feature_cols].to_numpy(dtype=np.float32)
        yval = df_val[target_col].to_numpy(dtype=np.float32)
        Xte  = df_test[feature_cols].to_numpy(dtype=np.float32)
        yte  = df_test[target_col].to_numpy(dtype=np.float32)

        params     = {} if lgb_params is None else dict(lgb_params)
        model      = build_lgbm_regressor(seed=seed, verbose=verbose_lgb, **params)
        eval_metric = "l1" if params.get("objective") == "regression_l1" else "rmse"

        if early_stopping_rounds > 0:
            model.fit(
                Xtr, ytr,
                eval_set=[(Xval, yval)],
                eval_metric=eval_metric,
                callbacks=[lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=False)],
            )
            best_it = getattr(model, "best_iteration_", None) or params.get("n_estimators", 100)
        else:
            # Pas d'early stopping : entraîner n_estimators fixe
            model.fit(Xtr, ytr)
            best_it = params.get("n_estimators", 100)

        pred_val = model.predict(Xval).astype(float)
        pred_te  = model.predict(Xte).astype(float)

        df_pred_val = df_val[["Date", "Ticker"]].copy()
        df_pred_val["y_true"] = yval.astype(float)
        df_pred_val["pred"]   = pred_val
        ic_val_s   = per_date_spearman_ic(df_pred_val)
        ic_val_p   = per_date_pearson_ic(df_pred_val)
        spread_val = per_date_long_short_spread(df_pred_val, q=q_spread)

        df_pred_te = df_test[["Date", "Ticker"]].copy()
        df_pred_te["y_true"] = yte.astype(float)
        df_pred_te["pred"]   = pred_te
        ic_te_s   = per_date_spearman_ic(df_pred_te)
        ic_te_p   = per_date_pearson_ic(df_pred_te)
        spread_te = per_date_long_short_spread(df_pred_te, q=q_spread)

        print(
            f"  Fold {int(i):>2} | {tr0.date()}>{tr1.date()} "
            f"| test {te0.date()}>{te1.date()}"
            f" | n_iter={best_it or 0:>4}"
            f" | val_IC={ic_val_s:+.3f}  test_IC={ic_te_s:+.3f}  LS={spread_te:+.4f}"
        )

        val_metric_last = np.nan
        try:
            val_metric_last = float(model.evals_result_["valid_0"][eval_metric][-1])
        except Exception:
            pass

        fold_rows.append({
            "fold_id": int(i),
            "train_start": tr0.date(), "train_end": tr1.date(),
            "test_start":  te0.date(), "test_end":  te1.date(),
            "status": "ok",
            "n_train": int(len(df_tr)), "n_val": int(len(df_val)), "n_test": int(len(df_test)),
            "best_iteration":  int(best_it) if best_it is not None else np.nan,
            "val_ic_spearman": ic_val_s, "val_ic_pearson": ic_val_p, "val_ls_spread": spread_val,
            "ic_spearman":     ic_te_s,  "ic_pearson":     ic_te_p,  "ls_spread":     spread_te,
            "val_metric_last": val_metric_last,
        })
        all_test_preds.append(df_pred_te)
        all_importances.append(
            pd.Series(model.feature_importances_, index=feature_cols, name=int(i))
        )

    fold_report = pd.DataFrame(fold_rows)
    df_all_test = pd.concat(all_test_preds, ignore_index=True) if all_test_preds else pd.DataFrame()
    return fold_report, df_all_test, pd.DataFrame(all_importances)


In [12]:
# =============================================================================
# 9. MODÈLE FINAL + RANKING DERNIÈRE DATE
# =============================================================================

def train_final_lgbm_and_rank_last_date(
    df_cs: pd.DataFrame,
    universe: str,
    feature_cols: list[str],
    target_col: str,
    seed: int = 42,
    early_stopping_rounds: int = 100,
    val_frac: float = 0.10,
    lgb_params: dict | None = None,
    verbose_lgb: int = -1,
):
    df_u = df_cs[df_cs["Universe"].astype(str) == universe].copy()
    df_u = df_u.sort_values(["Ticker", "Date"]).reset_index(drop=True)

    df_train_all = df_u.dropna(subset=[target_col]).copy()
    if len(df_train_all) < 500:
        raise ValueError(f"Pas assez de données d'entraînement ({len(df_train_all)} rows)")

    params      = {} if lgb_params is None else dict(lgb_params)
    eval_metric = "l1" if params.get("objective") == "regression_l1" else "rmse"
    best_it     = None

    # ── Early stopping pour trouver best_it ──
    if early_stopping_rounds and early_stopping_rounds > 0:
        unique_dates = np.sort(df_train_all["Date"].unique())
        if len(unique_dates) >= 20:
            k = max(1, int(np.floor(val_frac * len(unique_dates))))
            val_start_date = unique_dates[-k]
            is_val = df_train_all["Date"] >= val_start_date

            df_tr_es  = df_train_all[~is_val]
            df_val_es = df_train_all[ is_val]

            Xtr_es  = df_tr_es[feature_cols].to_numpy(dtype=np.float32)
            ytr_es  = df_tr_es[target_col].to_numpy(dtype=np.float32)
            Xval_es = df_val_es[feature_cols].to_numpy(dtype=np.float32)
            yval_es = df_val_es[target_col].to_numpy(dtype=np.float32)

            model_es = build_lgbm_regressor(seed=seed, verbose=verbose_lgb, **params)
            model_es.fit(
                Xtr_es, ytr_es,
                eval_set=[(Xval_es, yval_es)],
                eval_metric=eval_metric,
                callbacks=[lgb.early_stopping(
                    stopping_rounds=early_stopping_rounds, verbose=False
                )],
            )
            best_it = int(getattr(model_es, "best_iteration_", None) or model_es.n_estimators)
            print(f"[Final] early stopping → best_iteration={best_it}")

    # ── Refit sur TOUT ──
    Xtr_all = df_train_all[feature_cols].to_numpy(dtype=np.float32)
    ytr_all = df_train_all[target_col].to_numpy(dtype=np.float32)

    params_final = dict(params)
    if best_it is not None:
        params_final["n_estimators"] = best_it

    model = build_lgbm_regressor(seed=seed, verbose=verbose_lgb, **params_final)
    model.fit(Xtr_all, ytr_all)
    print(f"[Final] trained on {len(Xtr_all):,} rows, {len(feature_cols)} features")

    # ── Ranking sur la dernière date ──
    last_date = df_u["Date"].max()
    df_last   = df_u[df_u["Date"] == last_date].copy()

    if len(df_last) == 0:
        print(f"[Warn] Aucune donnée scorable à la dernière date ({last_date.date()})")
        return model, pd.DataFrame(), best_it

    preds = model.predict(df_last[feature_cols].to_numpy(dtype=np.float32)).astype(float)
    ranking = df_last[["Ticker", "Date"]].copy()
    ranking["pred_excess_return"] = preds
    ranking = ranking.sort_values("pred_excess_return", ascending=False).reset_index(drop=True)

    print(f"[Final] ranked {len(ranking)} tickers at {last_date.date()}")
    return model, ranking, best_it

In [13]:
# =============================================================================
# 10. RENDEMENTS RÉALISÉS (Yahoo Finance)
# =============================================================================

def fetch_realized_returns(
    ranking_df: pd.DataFrame, last_data_date: pd.Timestamp, horizon_days: int = 20
) -> pd.DataFrame:
    """
    Télécharge les prix Yahoo Finance depuis last_data_date,
    calcule le rendement réalisé et mesure la corrélation prédiction vs réalité.
    BUG FIX : corrige le cas start_date == end_date (date d'aujourd'hui).
    """
    tickers   = ranking_df["Ticker"].tolist()
    start_dl  = last_data_date.strftime("%Y-%m-%d")

    today = pd.Timestamp.today().normalize()
    horizon_end = last_data_date + timedelta(days=31)

    if horizon_end > today:
        # On est encore dans la fenêtre → utiliser hier comme fin
        end_dl = (today - timedelta(days=1)).strftime("%Y-%m-%d")
    else:
        end_dl = horizon_end.strftime("%Y-%m-%d")

    # ── Guard : start ne peut pas être >= end ──
    if pd.Timestamp(start_dl) >= pd.Timestamp(end_dl):
        print(f"[Yahoo] Pas assez de recul pour calculer les rendements réalisés "
              f"({start_dl} >= {end_dl}). Ranking renvoyé sans realized.")
        return ranking_df

    print(f"\n{'='*70}")
    print(f"[Yahoo] Downloading {len(tickers)} tickers | {start_dl} → {end_dl}")

    try:
        data = yf.download(
            tickers, start=start_dl, end=end_dl,
            auto_adjust=True, progress=False, threads=True,
        )
    except Exception as e:
        print(f"[Yahoo] FAILED: {e}")
        return ranking_df

    if data.empty:
        print("[Yahoo] No data returned")
        return ranking_df

    closes = data["Close"] if isinstance(data.columns, pd.MultiIndex) else data[["Close"]].rename(columns={"Close": tickers[0]})

    avail           = closes.index.sort_values()
    ref_candidates  = avail[avail >= last_data_date]
    if len(ref_candidates) == 0:
        print(f"[Yahoo] No dates >= {last_data_date.date()}")
        return ranking_df

    ref_date = ref_candidates[0]
    end_date = avail[-1]
    bdays    = len(avail[(avail >= ref_date) & (avail <= end_date)]) - 1
    print(f"[Yahoo] Ref: {ref_date.date()} | Latest: {end_date.date()} | "
          f"Bdays elapsed: {bdays}/{horizon_days}")

    rows = []
    for tk in tickers:
        if tk not in closes.columns:
            rows.append({"Ticker": tk, "price_start": np.nan, "price_end": np.nan,
                         "actual_return_%": np.nan, "_ok": False})
            continue
        s     = closes[tk].dropna()
        s_ref = s[s.index >= ref_date]
        if len(s_ref) < 2:
            rows.append({"Ticker": tk, "price_start": np.nan, "price_end": np.nan,
                         "actual_return_%": np.nan, "_ok": False})
            continue
        p0, p1 = float(s_ref.iloc[0]), float(s_ref.iloc[-1])
        rows.append({"Ticker": tk, "price_start": round(p0, 2), "price_end": round(p1, 2),
                     "actual_return_%": round(100 * (p1 / p0 - 1), 2), "_ok": True})

    df_act  = pd.DataFrame(rows)
    out     = ranking_df.merge(df_act, on="Ticker", how="left")
    med_act = df_act.loc[df_act["_ok"], "actual_return_%"].median()
    out["actual_excess_%"] = out["actual_return_%"] - med_act

    n_ok = df_act["_ok"].sum()
    print(f"[Yahoo] {n_ok}/{len(tickers)} tickers OK | Median actual return: {med_act:.2f}%")

    valid = out.dropna(subset=["pred_excess_%_approx", "actual_excess_%"])
    if len(valid) >= 10:
        ic_s, _ = spearmanr(valid["pred_excess_%_approx"], valid["actual_excess_%"])
        ic_p, _ = pearsonr( valid["pred_excess_%_approx"], valid["actual_excess_%"])
        print(f"[Yahoo] REALIZED IC: spearman={ic_s:.4f}  pearson={ic_p:.4f}")
        k       = max(1, int(np.floor(0.10 * len(valid))))
        sv      = valid.sort_values("pred_excess_%_approx", ascending=False)
        top_avg = sv.head(k)["actual_return_%"].mean()
        bot_avg = sv.tail(k)["actual_return_%"].mean()
        print(f"[Yahoo] REALIZED L/S: Top{k} avg={top_avg:.2f}% | Bot{k} avg={bot_avg:.2f}% | "
              f"Spread={top_avg - bot_avg:.2f}%")

    return out.drop(columns=["_ok"], errors="ignore")

In [14]:
# =============================================================================
# 11. PIPELINE COMPLÈTE PAR UNIVERS
# =============================================================================

def run_universe_pipeline(
    universe_name: str,
    df_cs: pd.DataFrame,
    folds: pd.DataFrame,
    feature_cols: list,
    lgb_params: dict,
    target_col: str = "excess_target_20d",
    q_spread: float = 0.10,
    seed: int = 42,
    val_frac_wf: float = 0.15,
    val_frac_final: float = 0.10,
    early_stopping_rounds: int = 100,
    min_train: int = 300,
    min_val: int = 50,
    min_test: int = 50,
    dashboard_template: str | None = None,
    dashboard_output:   str | None = None,
    open_browser: bool = True,
) -> dict:
    print(f"\n{'#'*90}")
    print(f"#  UNIVERS : {universe_name.upper()}  |  {len(feature_cols)} features  |  {len(folds)} folds")
    print(f"{'#'*90}")

    # ── 1. Préparer le df d'entraînement ──
    df_u_train = df_cs[df_cs["Universe"].astype(str) == universe_name].dropna(subset=[target_col])

    # ── 2. Walk-forward ──
    print(f"\n--- Walk-forward ---")
    fold_report, df_all_test, imp_df = walkforward_lgbm_rank(
        df_cs=df_cs, folds=folds, universe=universe_name,
        feature_cols=feature_cols, target_col=target_col, q_spread=q_spread,
        seed=seed, val_frac=val_frac_wf,
        early_stopping_rounds=early_stopping_rounds,
        lgb_params=lgb_params, verbose_lgb=-1,
        min_train=min_train, min_val=min_val, min_test=min_test,
    )

    # ── Résumé walk-forward ──
    ok = fold_report[fold_report["status"] == "ok"]
    print(f"\n--- Résumé folds ({len(ok)} OK / {len(fold_report)}) ---")
    if len(ok):
        desc = ok[["ic_spearman", "ic_pearson", "ls_spread"]].describe().loc[["mean","std","min","max"]].round(4)
        print(desc.to_string())

    # ── 3. Modèle final ──
    print(f"\n--- Modèle final ---")
    model_final, ranking, best_it = train_final_lgbm_and_rank_last_date(
        df_cs=df_cs, universe=universe_name,
        feature_cols=feature_cols, target_col=target_col,
        seed=seed, early_stopping_rounds=early_stopping_rounds,
        val_frac=val_frac_final, lgb_params=lgb_params,
    )

    # ── Conversion excess -> return absolu ──
    df_u = df_cs[df_cs["Universe"].astype(str) == universe_name].copy()
    df_u["Date"] = pd.to_datetime(df_u["Date"])
    raw_col = "target_20d"
    if raw_col in df_u.columns:
        med_series = (
            df_u.dropna(subset=[raw_col])
            .groupby("Date", sort=False, observed=True)[raw_col]
            .median().sort_index()
        )
        med_hat = float(med_series.rolling(60, min_periods=20).mean().iloc[-1]) if len(med_series) else 0.0
    else:
        med_hat = 0.0

    ranking = ranking.copy()
    ranking["pred_raw_log_hat"]     = ranking["pred_excess_return"] + med_hat
    ranking["pred_raw_ret_hat"]     = np.expm1(ranking["pred_raw_log_hat"])
    ranking["pred_raw_ret_hat_pct"] = 100 * ranking["pred_raw_ret_hat"]
    ranking["pred_excess_%_approx"] = 100 * np.expm1(ranking["pred_excess_return"])
    ranking["pred_abs_%_hat"]       = ranking["pred_raw_ret_hat_pct"]

    # ── 4. Rendements réalisés ──
    last_data_date = df_u["Date"].max()
    ranking = fetch_realized_returns(ranking, last_data_date, horizon_days=20)

    print("\n=== RANKING ===")
    cols = ["Ticker", "pred_abs_%_hat", "pred_excess_%_approx", "actual_return_%", "actual_excess_%"]
    cols = [c for c in cols if c in ranking.columns]
    print(ranking[cols].round(2).to_string(index=False))
    print(f"  med_hat = {med_hat*100:.2f}%")

    # ── 5. TABLE IC DES FEATURES ──
    print(f"\n--- Feature IC Table ({universe_name.upper()}) ---")
    feature_ic = show_feature_table(
        df=df_u_train,
        feature_cols=feature_cols,
        target_col=target_col,
    )

    # ── 6. Dashboard ──
    if dashboard_template and dashboard_output and generate_dashboard is not None:
        try:
            generate_dashboard(
                ranking=ranking, fold_report=fold_report,
                med_hat=med_hat, universe=universe_name,
                n_features=len(feature_cols), fold_report_raw="",
                imp_df=imp_df,
                template_path=dashboard_template, output_path=dashboard_output,
                open_browser=open_browser,
            )
        except Exception as e:
            print(f"[Dashboard] erreur : {e}")

    return {
        "model":       model_final,
        "ranking":     ranking,
        "fold_report": fold_report,
        "imp_df":      imp_df if not imp_df.empty else pd.DataFrame(),
        "feature_ic":  feature_ic,
        "med_hat":     med_hat,
        "best_it":     best_it,
    }


In [15]:
# =============================================================================
# 12. PRÉPARATION DES DONNÉES
# =============================================================================
# Les dates sont lues depuis UNIVERSE_CONFIG (défini en cellule 1).
# Pour changer les dates d'un univers, modifie UNIVERSE_CONFIG uniquement.
#
# Les features yf_*, temporelles et interactions sont calculées dans pipeline.py
# et stockées dans la table SQL 'features' de chaque DB (plus de .parquet).
# =============================================================================

TARGET_COL = "excess_target_20d"
Q_SPREAD   = 0.10

# ── Plage globale = union des plages de tous les univers ──
GLOBAL_START = min(c["start_date"]  for c in UNIVERSE_CONFIG.values())
# Si last_date est None, utiliser la date max disponible dans les données
for k, c in UNIVERSE_CONFIG.items():
    if c["last_date"] is None:
        from pipeline import _db_path, _max_date
        _db = _db_path(k)
        _px_max = _max_date(_db, "prices")
        if _px_max:
            c["last_date"] = str(_px_max)
            print(f"  {k}: last_date auto-détecté → {c['last_date']}")
        else:
            c["last_date"] = "2099-12-31"  # fallback — prend tout

GLOBAL_END   = max(c["last_date"]   for c in UNIVERSE_CONFIG.values())

# ── Lecture depuis les DBs SQLite (remplace les parquets) ──
df_raw = read_all_features()
df_raw = df_raw[
    (df_raw["Date"] >= pd.Timestamp(GLOBAL_START)) &
    (df_raw["Date"] <= pd.Timestamp(GLOBAL_END))
].copy()
df_raw = df_raw.sort_values(["Universe", "Ticker", "Date"], kind="mergesort").reset_index(drop=True)

# ── Targets ──
df_raw = add_forward_targets(df_raw, horizons=(5, 20), price_col="AdjClose")
df_raw, centers = add_excess_targets(df_raw, horizons=(5, 20), zscore=False)

df = df_raw.copy()
df["Universe"] = df["Universe"].astype("category")
df["Ticker"]   = df["Ticker"].astype("category")

yf_cols_present = [c for c in df.columns if c.startswith("yf_")]
tmp_cols = [c for c in df.columns if c.startswith("delta_") or c.startswith("ma_")]
macro_cols_present = [c for c in df.columns if c.startswith("macro_")]
print(f"\nDataframe : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"Features Yahoo  ({len(yf_cols_present)}) : {yf_cols_present}")
print(f"Features temporal ({len(tmp_cols)}) : {tmp_cols[:6]}{'...' if len(tmp_cols)>6 else ''}")
print(f"Features macro   ({len(macro_cols_present)}) : {macro_cols_present}"
      + ("" if macro_cols_present else " ⚠ absentes — lance pipeline.compute_features()"))


  spx: last_date auto-détecté → 2026-02-27
  cac40: last_date auto-détecté → 2026-02-27
  dax: last_date auto-détecté → 2026-02-27
  [S&P 500] 621,708 rows × 82 cols (503 tickers, 2021-03-29 → 2026-02-27)
  [CAC 40] 50,480 rows × 82 cols (40 tickers, 2021-03-29 → 2026-02-27)
  [DAX] 50,480 rows × 82 cols (40 tickers, 2021-03-29 → 2026-02-27)

Dataframe : 462,433 lignes x 86 colonnes
Features Yahoo  (17) : ['yf_dist_52w_high', 'yf_dist_52w_low', 'yf_rsi_14', 'yf_rsi_60', 'yf_vol_ratio', 'yf_vol_ratio_short', 'yf_volume_ratio', 'yf_dist_sma50', 'yf_dist_sma200', 'yf_mom_3m', 'yf_mom_6m', 'yf_mom_12m', 'yf_drawdown_252', 'yf_trend_consistency', 'yf_corr_idx_60', 'yf_beta_idx_252', 'yf_atr_ratio']
Features temporal (20) : ['delta_5d_mom_120', 'ma_5d_mom_120', 'delta_10d_mom_120', 'ma_10d_mom_120', 'delta_5d_rv_60', 'ma_5d_rv_60']...
Features macro   (3) : ['macro_vix', 'macro_vix_chg_20', 'macro_term_spread']


In [16]:
# =============================================================================
# 13. EXÉCUTION CAC40
# =============================================================================
cfg_cac = UNIVERSE_CONFIG["cac40"]

df_cac = df[
    (df["Universe"] == "cac40") &
    (df["Date"] >= pd.Timestamp(cfg_cac["start_date"])) &
    (df["Date"] <= pd.Timestamp(cfg_cac["last_date"]))
].copy()

print(f"CAC40 — {df_cac['Ticker'].nunique()} tickers | "
      f"{df_cac['Date'].min().date()} → {df_cac['Date'].max().date()}")

df_cs_cac, all_cols_cac = prepare_feature_df(df_cac)

EXCLUDE_CAC = {
    'range_hl_close',
    'ret_1',
    'ret_3',
    'mom_5',
    'mom_20',
    'mom_60',
    'mom_120',
    'dist_sma60',
    'slope_logp_60',
    'z_price_5',
    'rv_20',
    'rv_60',
    'vol_chg_vs20',
    'beta_60',
    'yf_dist_52w_high',
    'yf_dist_52w_low',
    'yf_rsi_60',
    'yf_volume_ratio',
    'yf_dist_sma50',
    'yf_dist_sma200',
    'yf_mom_3m',
    'yf_drawdown_252',
    'yf_beta_idx_252',
    'yf_atr_ratio',
    'delta_5d_mom_120',
    'ma_5d_mom_120',
    'ma_10d_mom_120',
    'ma_5d_rv_60',
    'delta_10d_rv_60',
    'ma_10d_rv_60',
    'ma_5d_dollar_volume',
    'delta_10d_dollar_volume',
    'ma_10d_dollar_volume',
    'delta_5d_beta_60',
    'ma_5d_beta_60',
    'ma_10d_beta_60',
    'ma_5d_slope_logp_60',
    'delta_10d_slope_logp_60',
    'ma_10d_slope_logp_60',
    'eps_surprise',
    'eps_surprise_abs',
    'eps_growth_qoq',
    'eps_surprise_ma4',
    'consecutive_beats',
    'revenue_growth_qoq',
    'revenue_growth_yoy',
    'net_margin',
    'debt_to_equity',
    'roa',
    'ocf_to_revenue'
}

feature_cols_cac = [c for c in all_cols_cac if c not in EXCLUDE_CAC]


print(f"  Features actives: {len(feature_cols_cac)} / {len(all_cols_cac)}"
      + (f" | exclusions: {sorted(EXCLUDE_CAC)}" if EXCLUDE_CAC else ""))

folds_cac = make_walkforward_folds(
    df=df_cs_cac.dropna(subset=[TARGET_COL]),
    start_date=cfg_cac["start_date"],
    holdout_start=cfg_cac["holdout_start"],
    train_years=cfg_cac["train_years"],
    test_months=cfg_cac["test_months"],
    step_months=cfg_cac["step_months"],
    embargo_bdays=cfg_cac["embargo_bdays"],
    date_col="Date",
)

lgb_params_cac = {
    "objective": "regression", "learning_rate": 0.03, "n_estimators": 5000,
    "num_leaves": 12, "max_depth": 5, "min_child_samples": 100,
    "min_split_gain": 0.001, "subsample": 0.5, "subsample_freq": 1,
    "colsample_bytree": 0.5, "feature_fraction_bynode": 0.5,
    "reg_alpha": 1.0, "reg_lambda": 10.0, "max_bin": 127,
    "min_data_in_bin": 10, "extra_trees": True,
}

results_cac = run_universe_pipeline(
    universe_name="cac40",
    df_cs=df_cs_cac, folds=folds_cac,
    feature_cols=feature_cols_cac,
    lgb_params=lgb_params_cac,
    target_col=TARGET_COL, q_spread=Q_SPREAD,
    val_frac_wf=cfg_cac["val_frac_wf"],
    val_frac_final=cfg_cac["val_frac_final"],
    dashboard_template="cac40_dashboard_template.html",
    dashboard_output="cac40_dashboard.html",
    open_browser=True,
)


CAC40 — 40 tickers | 2023-01-02 → 2026-02-27
  Base: 73 (tech_cs=19, fund=12, macro=3)
  Temporal: 20  Interactions: 2  Yahoo: 17  Macro: 3
  => 73 features disponibles au total
  Features actives: 23 / 73 | exclusions: ['beta_60', 'consecutive_beats', 'debt_to_equity', 'delta_10d_dollar_volume', 'delta_10d_rv_60', 'delta_10d_slope_logp_60', 'delta_5d_beta_60', 'delta_5d_mom_120', 'dist_sma60', 'eps_growth_qoq', 'eps_surprise', 'eps_surprise_abs', 'eps_surprise_ma4', 'ma_10d_beta_60', 'ma_10d_dollar_volume', 'ma_10d_mom_120', 'ma_10d_rv_60', 'ma_10d_slope_logp_60', 'ma_5d_beta_60', 'ma_5d_dollar_volume', 'ma_5d_mom_120', 'ma_5d_rv_60', 'ma_5d_slope_logp_60', 'mom_120', 'mom_20', 'mom_5', 'mom_60', 'net_margin', 'ocf_to_revenue', 'range_hl_close', 'ret_1', 'ret_3', 'revenue_growth_qoq', 'revenue_growth_yoy', 'roa', 'rv_20', 'rv_60', 'slope_logp_60', 'vol_chg_vs20', 'yf_atr_ratio', 'yf_beta_idx_252', 'yf_dist_52w_high', 'yf_dist_52w_low', 'yf_dist_sma200', 'yf_dist_sma50', 'yf_drawdown_2

In [17]:
# =============================================================================
# FEATURE SELECTION — CAC40
# =============================================================================
# RUN_DIAGNOSE_CAC = True  → hill-climb depuis le set actuel (EXCLUDE_CAC)
# =============================================================================

RUN_DIAGNOSE_CAC = False   # ← True pour améliorer le set actuel

if not RUN_DIAGNOSE_CAC:
    print("  [CAC40] Feature selection skippée — EXCLUDE_CAC actif")
    print(f"  feature_cols_cac : {len(feature_cols_cac)} features")
else:
    _df_u_cac = df_cs_cac[df_cs_cac['Universe'].astype(str) == 'cac40'].copy()
    _df_u_cac = _df_u_cac.sort_values(['Ticker', 'Date']).reset_index(drop=True)
    fold_arrays_cac = _prepare_fold_arrays_fs(
        _df_u_cac, folds_cac, all_cols_cac, TARGET_COL,
        val_frac=cfg_cac["val_frac_wf"], min_train=300, min_val=50, min_test=50,
    )

    improved_cac, diag_history_cac = diagnose_feature_set(
        df_cs=df_cs_cac, folds=folds_cac, universe="cac40",
        current_features=feature_cols_cac,
        all_features=all_cols_cac,
        target_col=TARGET_COL, lgb_params=lgb_params_cac,
        metric="ic_ir",
        fold_arrays=fold_arrays_cac,
        early_stopping_rounds=100,
        max_rounds=20,
    )


  [CAC40] Feature selection skippée — EXCLUDE_CAC actif
  feature_cols_cac : 23 features


In [18]:
# =============================================================================
# 15. EXÉCUTION DAX
# =============================================================================
cfg_dax = UNIVERSE_CONFIG["dax"]

df_dax = df[
    (df["Universe"] == "dax") &
    (df["Date"] >= pd.Timestamp(cfg_dax["start_date"])) &
    (df["Date"] <= pd.Timestamp(cfg_dax["last_date"]))
].copy()

print(f"DAX — {df_dax['Ticker'].nunique()} tickers | "
      f"{df_dax['Date'].min().date()} → {df_dax['Date'].max().date()}")

df_cs_dax, all_cols_dax = prepare_feature_df(df_dax)

EXCLUDE_DAX = {
   'dollar_volume',
    'ret_1',
    'ret_2',
    'ret_3',
    'mom_5',
    'mom_20',
    'mom_60',
    'mom_120',
    'dist_sma20',
    'dist_sma60',
    'slope_logp_20',
    'slope_logp_60',
    'z_price_5',
    'z_price_10',
    'rv_20',
    'rv_60',
    'vol_chg_vs20',
    'yf_dist_52w_high',
    'yf_dist_52w_low',
    'yf_rsi_14',
    'yf_rsi_60',
    'yf_vol_ratio',
    'yf_vol_ratio_short',
    'yf_volume_ratio',
    'yf_dist_sma50',
    'yf_dist_sma200',
    'yf_mom_3m',
    'yf_mom_6m',
    'yf_mom_12m',
    'yf_drawdown_252',
    'yf_trend_consistency',
    'yf_beta_idx_252',
    'ma_5d_mom_120',
    'delta_10d_mom_120',
    'ma_10d_mom_120',
    'ma_5d_rv_60',
    'delta_10d_rv_60',
    'ma_10d_rv_60',
    'delta_5d_dollar_volume',
    'ma_10d_dollar_volume',
    'delta_5d_beta_60',
    'ma_5d_beta_60',
    'delta_10d_beta_60',
    'ma_10d_beta_60',
    'delta_5d_slope_logp_60',
    'ma_5d_slope_logp_60',
    'delta_10d_slope_logp_60',
    'mom20_div_mom120',
    'eps_surprise',
    'eps_surprise_ma4',
    'revenue_growth_yoy',
    'net_margin',
    'debt_to_equity',
    'roa',
    'ocf_to_revenue'
}

feature_cols_dax = [c for c in all_cols_dax if c not in EXCLUDE_DAX]


print(f"  Features actives: {len(feature_cols_dax)} / {len(all_cols_dax)}"
      + (f" | exclusions: {sorted(EXCLUDE_DAX)}" if EXCLUDE_DAX else ""))

folds_dax = make_walkforward_folds(
    df=df_cs_dax.dropna(subset=[TARGET_COL]),
    start_date=cfg_dax["start_date"],
    holdout_start=cfg_dax["holdout_start"],
    train_years=cfg_dax["train_years"],
    test_months=cfg_dax["test_months"],
    step_months=cfg_dax["step_months"],
    embargo_bdays=cfg_dax["embargo_bdays"],
    date_col="Date",
)

lgb_params_dax = {
    "objective": "regression", "learning_rate": 0.03,
    "n_estimators": 200,                # fixe — même approche que SPX (ES désactivé)
    "num_leaves": 12, "max_depth": 5, "min_child_samples": 100,
    "min_split_gain": 0.001, "subsample": 0.5, "subsample_freq": 1,
    "colsample_bytree": 0.5, "feature_fraction_bynode": 0.5,
    "reg_alpha": 1.0, "reg_lambda": 10.0, "max_bin": 127,
    "min_data_in_bin": 10, "extra_trees": True,
}

results_dax = run_universe_pipeline(
    universe_name="dax",
    df_cs=df_cs_dax, folds=folds_dax,
    feature_cols=feature_cols_dax,
    lgb_params=lgb_params_dax,
    target_col=TARGET_COL, q_spread=Q_SPREAD,
    val_frac_wf=cfg_dax["val_frac_wf"],
    val_frac_final=cfg_dax["val_frac_final"],
    early_stopping_rounds=0,  # DÉSACTIVÉ — même diagnostic que SPX (val anti-corrélée au test)
    dashboard_template="dax_dashboard_template.html",
    dashboard_output="dax_dashboard.html",
    open_browser=True,
)


DAX — 40 tickers | 2023-01-02 → 2026-02-27
  Base: 73 (tech_cs=19, fund=12, macro=3)
  Temporal: 20  Interactions: 2  Yahoo: 17  Macro: 3
  => 73 features disponibles au total
  Features actives: 18 / 73 | exclusions: ['debt_to_equity', 'delta_10d_beta_60', 'delta_10d_mom_120', 'delta_10d_rv_60', 'delta_10d_slope_logp_60', 'delta_5d_beta_60', 'delta_5d_dollar_volume', 'delta_5d_slope_logp_60', 'dist_sma20', 'dist_sma60', 'dollar_volume', 'eps_surprise', 'eps_surprise_ma4', 'ma_10d_beta_60', 'ma_10d_dollar_volume', 'ma_10d_mom_120', 'ma_10d_rv_60', 'ma_5d_beta_60', 'ma_5d_mom_120', 'ma_5d_rv_60', 'ma_5d_slope_logp_60', 'mom20_div_mom120', 'mom_120', 'mom_20', 'mom_5', 'mom_60', 'net_margin', 'ocf_to_revenue', 'ret_1', 'ret_2', 'ret_3', 'revenue_growth_yoy', 'roa', 'rv_20', 'rv_60', 'slope_logp_20', 'slope_logp_60', 'vol_chg_vs20', 'yf_beta_idx_252', 'yf_dist_52w_high', 'yf_dist_52w_low', 'yf_dist_sma200', 'yf_dist_sma50', 'yf_drawdown_252', 'yf_mom_12m', 'yf_mom_3m', 'yf_mom_6m', 'yf_rs

In [19]:
# =============================================================================
# FEATURE SELECTION — DAX
# =============================================================================

RUN_DIAGNOSE_DAX = False

if not RUN_DIAGNOSE_DAX:
    print("  [DAX] Feature selection skippée — EXCLUDE_DAX actif")
    print(f"  feature_cols_dax : {len(feature_cols_dax)} features")
else:
    _df_u_dax = df_cs_dax[df_cs_dax['Universe'].astype(str) == 'dax'].copy()
    _df_u_dax = _df_u_dax.sort_values(['Ticker', 'Date']).reset_index(drop=True)
    fold_arrays_dax = _prepare_fold_arrays_fs(
        _df_u_dax, folds_dax, all_cols_dax, TARGET_COL,
        val_frac=cfg_dax["val_frac_wf"], min_train=300, min_val=50, min_test=50,
    )

    improved_dax, diag_history_dax = diagnose_feature_set(
        df_cs=df_cs_dax, folds=folds_dax, universe="dax",
        current_features=feature_cols_dax,
        all_features=all_cols_dax,
        target_col=TARGET_COL, lgb_params=lgb_params_dax,
        metric="ic_ir",
        fold_arrays=fold_arrays_dax,
        early_stopping_rounds=0,  # désactivé pour DAX
        max_rounds=20,
    )


  [DAX] Feature selection skippée — EXCLUDE_DAX actif
  feature_cols_dax : 18 features


In [20]:
# =============================================================================
# 16. EXÉCUTION SPX
# =============================================================================
cfg_spx = UNIVERSE_CONFIG["spx"]

# ── Filtrage dates ──
df_spx = df[
    (df["Universe"] == "spx") &
    (df["Date"] >= pd.Timestamp(cfg_spx["start_date"])) &
    (df["Date"] <= pd.Timestamp(cfg_spx["last_date"]))
].copy()

print(f"SPX — {df_spx['Ticker'].nunique()} tickers | "
      f"{df_spx['Date'].min().date()} → {df_spx['Date'].max().date()}")

# ── Générer TOUTES les features ──
df_cs_spx, all_cols_spx = prepare_feature_df(df_spx)

# ── Exclure explicitement les features non désirées ──
EXCLUDE_SPX = { 

    'range_hl_close',
    'ret_1',
    'ret_2',
    'ret_3',
    'mom_5',
    'mom_20',
    'mom_60',
    'mom_120',
    'dist_sma60',
    'slope_logp_20',
    'z_price_5',
    'z_price_10',
    'rv_20',
    'vol_chg_vs20',
    'beta_60',
    'yf_dist_52w_high',
    'yf_rsi_14',
    'yf_vol_ratio',
    'yf_volume_ratio',
    'yf_dist_sma50',
    'yf_dist_sma200',
    'yf_mom_3m',
    'yf_mom_6m',
    'yf_drawdown_252',
    'yf_trend_consistency',
    'yf_corr_idx_60',
    'yf_beta_idx_252',
    'yf_atr_ratio',
    'macro_vix',
    'delta_5d_mom_120',
    'ma_5d_mom_120',
    'delta_10d_mom_120',
    'ma_10d_mom_120',
    'delta_5d_rv_60',
    'ma_5d_rv_60',
    'ma_10d_rv_60',
    'delta_5d_dollar_volume',
    'ma_5d_dollar_volume',
    'delta_10d_dollar_volume',
    'ma_10d_dollar_volume',
    'delta_5d_beta_60',
    'ma_5d_beta_60',
    'delta_10d_beta_60',
    'ma_10d_beta_60',
    'ma_5d_slope_logp_60',
    'delta_10d_slope_logp_60',
    'ma_10d_slope_logp_60',
    'mom120_div_rv60',
    'mom20_div_mom120',
    'eps_surprise',
    'eps_surprise_abs',
    'eps_beat',
    'eps_growth_qoq',
    'eps_surprise_ma4',
    'revenue_growth_qoq',
    'revenue_growth_yoy',
    'net_margin',
    'debt_to_equity',
    'roa',
    'ocf_to_revenue'
}

feature_cols_spx = [c for c in all_cols_spx if c not in EXCLUDE_SPX]

        
print(f"  Features actives: {len(feature_cols_spx)} / {len(all_cols_spx)}"
      + (f" | exclusions: {sorted(EXCLUDE_SPX)}" if EXCLUDE_SPX else ""))

# ── Walk-forward ──
folds_spx = make_walkforward_folds(
    df=df_cs_spx.dropna(subset=[TARGET_COL]),
    start_date=cfg_spx["start_date"],
    holdout_start=cfg_spx["holdout_start"],
    train_years=cfg_spx["train_years"],
    test_months=cfg_spx["test_months"],
    step_months=cfg_spx["step_months"],
    embargo_bdays=cfg_spx["embargo_bdays"],
    date_col="Date",
)

lgb_params_spx = {
    # Params CAC40 (qui marchent) + min_child_samples scalé ×(503/40)
    # + n_estimators FIXE (pas d'early stopping — la validation SPX est anti-corrélée au test)
    "objective": "regression", "learning_rate": 0.03,
    "n_estimators": 150,                # fixe — folds à 50 iter marchent mieux que ceux à 600+
    "num_leaves": 12, "max_depth": 5,
    "min_child_samples": 1250,          # scalé ×(503/40) = même ratio dates/feuille que CAC40
    "min_split_gain": 0.001, "subsample": 0.5, "subsample_freq": 1,
    "colsample_bytree": 0.5, "feature_fraction_bynode": 0.5,
    "reg_alpha": 1.0, "reg_lambda": 10.0, "max_bin": 127,
    "min_data_in_bin": 10, "extra_trees": True,
}

results_spx = run_universe_pipeline(
    universe_name="spx",
    df_cs=df_cs_spx, folds=folds_spx,
    feature_cols=feature_cols_spx,
    lgb_params=lgb_params_spx,
    target_col=TARGET_COL, q_spread=Q_SPREAD,
    val_frac_wf=cfg_spx["val_frac_wf"],
    val_frac_final=cfg_spx["val_frac_final"],
    early_stopping_rounds=0,  # DÉSACTIVÉ — validation anti-corrélée au test sur SPX
    dashboard_template="spx_dashboard_template.html",
    dashboard_output="spx_dashboard.html",
    open_browser=True,
)


SPX — 503 tickers | 2023-01-03 → 2026-02-27
  Base: 73 (tech_cs=19, fund=12, macro=3)
  Temporal: 20  Interactions: 2  Yahoo: 17  Macro: 3
  => 73 features disponibles au total
  Features actives: 13 / 73 | exclusions: ['beta_60', 'debt_to_equity', 'delta_10d_beta_60', 'delta_10d_dollar_volume', 'delta_10d_mom_120', 'delta_10d_slope_logp_60', 'delta_5d_beta_60', 'delta_5d_dollar_volume', 'delta_5d_mom_120', 'delta_5d_rv_60', 'dist_sma60', 'eps_beat', 'eps_growth_qoq', 'eps_surprise', 'eps_surprise_abs', 'eps_surprise_ma4', 'ma_10d_beta_60', 'ma_10d_dollar_volume', 'ma_10d_mom_120', 'ma_10d_rv_60', 'ma_10d_slope_logp_60', 'ma_5d_beta_60', 'ma_5d_dollar_volume', 'ma_5d_mom_120', 'ma_5d_rv_60', 'ma_5d_slope_logp_60', 'macro_vix', 'mom120_div_rv60', 'mom20_div_mom120', 'mom_120', 'mom_20', 'mom_5', 'mom_60', 'net_margin', 'ocf_to_revenue', 'range_hl_close', 'ret_1', 'ret_2', 'ret_3', 'revenue_growth_qoq', 'revenue_growth_yoy', 'roa', 'rv_20', 'slope_logp_20', 'vol_chg_vs20', 'yf_atr_ratio'

In [21]:
# =============================================================================
# FEATURE SELECTION — SPX
# =============================================================================

RUN_DIAGNOSE_SPX = False

if not RUN_DIAGNOSE_SPX:
    print("  [SPX] Feature selection skippée — EXCLUDE_SPX actif")
    print(f"  feature_cols_spx : {len(feature_cols_spx)} features")
else:
    _df_u_spx = df_cs_spx[df_cs_spx['Universe'].astype(str) == 'spx'].copy()
    _df_u_spx = _df_u_spx.sort_values(['Ticker', 'Date']).reset_index(drop=True)
    fold_arrays_spx = _prepare_fold_arrays_fs(
        _df_u_spx, folds_spx, all_cols_spx, TARGET_COL,
        val_frac=cfg_spx["val_frac_wf"], min_train=300, min_val=50, min_test=50,
    )

    improved_spx, diag_history_spx = diagnose_feature_set(
        df_cs=df_cs_spx, folds=folds_spx, universe="spx",
        current_features=feature_cols_spx,
        all_features=all_cols_spx,
        target_col=TARGET_COL, lgb_params=lgb_params_spx,
        metric="ic_ir",
        fold_arrays=fold_arrays_spx,
        early_stopping_rounds=0,  # désactivé pour SPX
        max_rounds=20,
    )


  [SPX] Feature selection skippée — EXCLUDE_SPX actif
  feature_cols_spx : 13 features
